In [6]:
from datasets import load_dataset

In [7]:
!pip install -U datasets

In [8]:
import tensorflow as tf

for gpu in tf.config.list_physical_devices("GPU"):
    print(gpu)

PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')


In [9]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [ ]:
import requests

for url in [
    "https://huggingface.co",
    "https://pypi.org"
]:
    try:
        response = requests.get(url, timeout=10)
        print(url, response.status_code)
    except Exception as e:
        print(url, type(e).__name__, str(e))

https://huggingface.co ConnectionError HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='huggingface.co', port=443): Failed to resolve 'huggingface.co' ([Errno -3] Temporary failure in name resolution)"))


In [ ]:
dataset=load_dataset("Helsinki-NLP/opus_books", "en-fr")

In [ ]:
print(dataset)

In [ ]:
train_data=dataset['train']
print(train_data)

In [ ]:
def find_duplicates(data):
  seen_data=set()
  cleaned_data=[]
  duplicates_counts=0
  for example in data:
    fr=example['translation']['fr']
    en=example['translation']['en']
    if not en or not fr:
      continue
    pair=(en,fr)
    if pair not in seen_data:
      seen_data.add(pair)
      cleaned_data.append({"en":en,"fr":fr})
    else:
      duplicates_counts+=1
  return cleaned_data,duplicates_counts


In [ ]:
cleaned_data,duplicates_counts=find_duplicates(dataset['train'])
print(duplicates_counts)
len(cleaned_data)


In [ ]:
type(cleaned_data)

In [ ]:

import tensorflow as tf

en = [x["en"] for x in cleaned_data]
fr = [x["fr"] for x in cleaned_data]

# Keep every English–French pair together.
cleaned_data = tf.data.Dataset.from_tensor_slices((en, fr))

dataset_size = len(en)

# Shuffle once so separate iterations keep the same order.
cleaned_data = cleaned_data.shuffle(
    buffer_size=dataset_size,
    seed=42,
    reshuffle_each_iteration=False
)

training_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)

training_data = cleaned_data.take(training_size)
remaining_data = cleaned_data.skip(training_size)

val_data = remaining_data.take(val_size)
test_data = remaining_data.skip(val_size)


In [ ]:
en_train=training_data.map(lambda en,fr:en)
fr_train=training_data.map(lambda en,fr:fr)
en_val=val_data.map(lambda en,fr:en)
fr_val=val_data.map(lambda en,fr:fr)
en_test=test_data.map(lambda en,fr:en)
fr_test=test_data.map(lambda en,fr:fr)



In [ ]:
def add_english_token(eng):
  start="SOS "
  return tf.strings.join([start,eng])
def add_french_token(fr):
   start="SOS "
   end=" EOS"
   return tf.strings.join([start,fr,end])

In [ ]:
training_eng=en_train.map(add_english_token)
training_frn=fr_train.map(add_french_token)
val_eng=en_val.map(add_english_token)
val_frn=fr_val.map(add_french_token)
test_eng=en_test.map(add_english_token)
test_frn = fr_test.map(add_french_token)

In [ ]:
for sentence in training_eng.take(1):
  print(sentence.numpy().decode("utf-8"))
for sentence in val_frn.take(1):
  print(sentence.numpy().decode("utf-8"))


In [ ]:
vocab_size=100000
max_length=60
eng_vectorizer=tf.keras.layers.TextVectorization(max_tokens=vocab_size,standardize="lower_and_strip_punctuation",output_mode='int',split="whitespace",output_sequence_length=max_length)
frn_vectorizer=tf.keras.layers.TextVectorization(max_tokens=vocab_size,standardize="lower_and_strip_punctuation",output_mode='int',split="whitespace",output_sequence_length=max_length)

eng_vectorizer.adapt(training_eng.batch(128))
frn_vectorizer.adapt(training_frn.batch(128))

In [ ]:
print(frn_vectorizer.get_vocabulary()[0:5])
eng_vectorizer.get_vocabulary()[0:5]

In [ ]:
training_eng=training_eng.map(eng_vectorizer)
val_eng=val_eng.map(eng_vectorizer)
test_eng=test_eng.map(eng_vectorizer)
training_frn=training_frn.map(frn_vectorizer)
val_frn=val_frn.map(frn_vectorizer)
test_frn=test_frn.map(frn_vectorizer)



In [ ]:
sentence=next(iter(training_eng.take(1)))
print(sentence.shape)
sentence=next(iter(training_frn.take(1)))
print(sentence.shape)


In [ ]:
class Encoder(tf.keras.layers.Layer):
  def __init__(self,vocab_size,embedding_size,units):
    super().__init__()
    self.embedding=tf.keras.layers.Embedding(input_dim=vocab_size,output_dim=embedding_size,mask_zero=True)
    self.lstm=tf.keras.layers.LSTM(units=units,return_sequences=True,return_state=False)

  def call(self,input):
      x=self.embedding(input)
      output=self.lstm(x)
      return output


In [ ]:
class Crossattention(tf.keras.layers.Layer):
  def __init__(self,units):
    super().__init__()
    self.mha=tf.keras.layers.MultiHeadAttention(num_heads=1,key_dim=units )
    self.add=tf.keras.layers.Add()
    self.normaliz=tf.keras.layers.LayerNormalization()
  def call(self,context,target,pad_mask):
    attention_mask=None
    if pad_mask is not None:
      attention_mask = pad_mask[:, tf.newaxis, :]
    output=self.mha(query=target,value=context,key=context,attention_mask=attention_mask)
    output=self.add([target,output])
    output=self.normaliz(output)
    return output

In [ ]:
class Decoder(tf.keras.layers.Layer):
  def __init__(self,vocab_size,embedding_dims,units):
    super().__init__()
    self.embedding=tf.keras.layers.Embedding(input_dim=vocab_size,output_dim=embedding_dims,mask_zero=True)
    self.pre_attention_lstm=tf.keras.layers.LSTM(units=units,return_sequences=True,return_state=False)
    self.attention=Crossattention(units)
    self.post_attention_lstm=tf.keras.layers.LSTM(units=units,return_sequences=True,return_state=False)
    self.dense=tf.keras.layers.Dense(units=vocab_size)
  def call(self,source,target,source_mask):
    x=self.embedding(target)
    decoder_mask = tf.not_equal(target, 0)
    x=self.pre_attention_lstm(x,mask=decoder_mask)
    x=self.attention(source,x,pad_mask=source_mask)
    x=self.post_attention_lstm(x,mask=decoder_mask)
    output=self.dense(x)
    return output

In [ ]:
class Translate(tf.keras.Model):
  def __init__(self,units,embedding_dims,vocab_size):
    super().__init__()
    self.encoder=Encoder(vocab_size,embedding_dims,units)
    self.decoder=Decoder(vocab_size,embedding_dims,units)
  def call(self,inputs):
    english_ids,french_ids=inputs
    source_mask = tf.not_equal(english_ids, 0)
    context=self.encoder(english_ids)
    output=self.decoder(context,french_ids,source_mask)
    return output

In [ ]:
loss_fn=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True,reduction='none')
def masked_loss(ytrue,ypred):
  loss=loss_fn(ytrue,ypred)
  mask=tf.cast(ytrue!=0,loss.dtype)
  loss = loss * mask
  return tf.reduce_sum(loss) / tf.reduce_sum(mask)

In [ ]:
vocab_size=10000
model=Translate(256,256,vocab_size)

In [ ]:
with strategy.scope():
    model = Translate(
        units=units,
        embedding_dims=embedding_dims,
        eng_vocab_size=len(eng_vectorizer.get_vocabulary()),
        frn_vocab_size=len(frn_vectorizer.get_vocabulary())
    )
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=masked_loss,
    metrics=["accuracy"]
)

In [ ]:
def prepare_data(english, french):
    frn_input = french[:-1]
    frn_target = french[1:]
    return (english, frn_input), frn_target

training_data = tf.data.Dataset.zip(
    (training_eng, training_frn)
).map(prepare_data).batch(64)
val_data = tf.data.Dataset.zip(
    (val_eng, val_frn)
).map(prepare_data).batch(64)

In [ ]:
model.fit(training_data,validation_data=val_data,epochs=15)

In [ ]:

def translate(model, sentence, eng_vectorizer, frn_vectorizer,
              max_length=50):
    # Convert the English sentence into token IDs.
    english_ids = eng_vectorizer(
        tf.constant([sentence])
    )
    # Obtain the English encoder output.
    source_mask = tf.not_equal(english_ids, 0)
    context = model.encoder(english_ids)

    # Find the start and end token IDs. 
    vocab = frn_vectorizer.get_vocabulary()

    start_candidates = ["[start]", "<start>", "sos"]
    end_candidates = ["[end]", "<end>", "eos"]

    start_token = next(
        (token for token in start_candidates if token in vocab),
        None
    )
    end_token = next(
        (token for token in end_candidates if token in vocab),
        None
    )

    if start_token is None or end_token is None:
        raise ValueError(
            "Could not find start/end tokens. Check your French "
            "vectorizer vocabulary and replace the candidate tokens."
        )

    start_id = vocab.index(start_token)
    end_id = vocab.index(end_token)

    # Begin the French sequence with the start token.
    generated = [start_id]

    for _ in range(max_length):
        target_ids = tf.constant([generated], dtype=tf.int64)

        # Run the decoder on tokens generated so far.
        logits = model.decoder(
            context,
            target_ids,
            source_mask
        )

        # Predict the next token from the last position.
        next_id = int(
            tf.argmax(logits[0, -1, :]).numpy()
        )

        if next_id == end_id:
            break

        generated.append(next_id)

    # Convert token IDs back to French words.
    return " ".join(
        vocab[token_id]
        for token_id in generated[1:]
        if token_id < len(vocab) and vocab[token_id] != ""
    )


In [ ]:
sentence = "I am a student."
translation = translate(model, sentence, eng_vectorizer, frn_vectorizer)
print(f"English: {sentence}")   
print(f"French: {translation}")

In [ ]:
print("hello")